In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append('../../')  # Add project root to path

from datetime import datetime
import logging
from modules.base_ingestion import BaseBronzeIngestion
from typing import List, Dict, Any

# Log Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class OrgaosEventosIngestion(BaseBronzeIngestion):
    """Concrete implementation for Orgaos/Eventos ingestion."""

    def __init__(self, spark, entity_name: str = 'orgaos_eventos'):
        """Initialize calling base class"""
        # Pass spark and entity name to initialize base class
        super().__init__(
            spark=spark,
            entity_name=entity_name
        )
    
    def fetch_data(self) -> List[Dict[str, Any]]:
        """Implements the abstract method - uses _fetch_multithreaded from base class."""
        logger.info(f"Starting {self.entity_config} ingestion pipeline...")

        current_year = datetime.now().year
        params = self.entity_config['params'].copy()
        
        # Use multithreaded extraction to fetch members for each frente
        return self._fetch_multithreaded(
            source_table=f"{self.generic_config['catalog']}.camara_{self.generic_config['layer']}.orgaos",
            id_column='id',
            endpoint_builder=lambda id: f"orgaos/{id}/eventos",
            foreign_key='id_frente',
            params=params,  # param for fetch file from repository
        )

# Create ingestion instance
ingestion = OrgaosEventosIngestion(spark=spark)

In [0]:
try:
    ingestion.execute()
    logger.info(f"{ingestion.config.entity} ingestion completed successfully!")
except Exception as e:
    logger.error(f"{ingestion.config.entity} ingestion failed: {str(e)}")
    raise

In [0]:
%sql 
SELECT count(*) FROM workspace.camara_bronze.orgaos_eventos

In [0]:
%sql 
SELECT *
FROM workspace.camara_bronze.orgaos_eventos